# NOAA Feature Analysis

Compare feature correlations across two snapshots:
- **Pre-COVID**: December 2019
- **Latest**: Most recent snapshot

Indices:
- **Economic Damage**: NOAA_Economic_Damage_Index
- **Human Impact**: NOAA_Human_Impact_Index

In [16]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
import warnings
warnings.filterwarnings('ignore')

# Add project root to path
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from settings import DATA_PATH
from utils.transforms import winsorize_covid_period

# Styling
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 10
pd.set_option('display.max_rows', 200)
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

## 1. Load Data

In [17]:
# --- Paths ---
NOAA_DIR = DATA_PATH / "Exogenous_data" / "exogenous_noaa_snapshots" / "decades"
TARGET_DIR = DATA_PATH / "NFP_target"

# --- Load pre-COVID snapshot (Dec 2019) ---
pre_covid_path = NOAA_DIR / "2010s" / "2019" / "2019-12.parquet"
snap_pre = pd.read_parquet(pre_covid_path)
snap_pre['date'] = pd.to_datetime(snap_pre['date'])
print(f"Pre-COVID snapshot: {snap_pre['series_name'].nunique()} series, "
      f"{snap_pre['date'].min().date()} to {snap_pre['date'].max().date()}")

# --- Find and load latest snapshot ---
latest_files = sorted(NOAA_DIR.rglob('*.parquet'))
latest_path = latest_files[-1]
snap_latest = pd.read_parquet(latest_path)
snap_latest['date'] = pd.to_datetime(snap_latest['date'])
print(f"Latest snapshot: {latest_path.stem} — {snap_latest['series_name'].nunique()} series, "
      f"{snap_latest['date'].min().date()} to {snap_latest['date'].max().date()}")

# --- Load NFP target data ---
target_nsa = pd.read_parquet(TARGET_DIR / "total_nsa_first_release.parquet")
target_sa  = pd.read_parquet(TARGET_DIR / "total_sa_first_release.parquet")

# Compute MoM difference and acceleration
for df in [target_nsa, target_sa]:
    df['ds'] = pd.to_datetime(df['ds'])
    df.sort_values('ds', inplace=True)
    df['y_mom'] = df['y'].diff()
    df['y_acc'] = df['y_mom'].diff()

target_nsa = target_nsa.dropna(subset=['y_mom']).set_index('ds')
target_sa  = target_sa.dropna(subset=['y_mom']).set_index('ds')

print(f"\nNSA target: {len(target_nsa)} months ({target_nsa.index.min().date()} to {target_nsa.index.max().date()})")
print(f"SA  target: {len(target_sa)} months ({target_sa.index.min().date()} to {target_sa.index.max().date()})")

Pre-COVID snapshot: 192 series, 1990-01-01 to 2019-08-01
Latest snapshot: 2026-03 — 192 series, 1990-01-01 to 2025-11-01

NSA target: 432 months (1990-02-01 to 2026-01-01)
SA  target: 432 months (1990-02-01 to 2026-01-01)


In [18]:
# --- 1b. Drop Short-History Features ---
MIN_VALID_OBS = 60

valid_counts_latest = snap_latest.groupby('series_name')['value'].count()
valid_counts_pre = snap_pre.groupby('series_name')['value'].count()

short_features = valid_counts_latest[valid_counts_latest < MIN_VALID_OBS].sort_values()

if len(short_features) > 0:
    print(f"⚠ FLAGGED: {len(short_features)} features with < {MIN_VALID_OBS} valid data points (dropped from analysis)")
    print("  These are typically rare-event indicators or very new series.\n")
    
    flagged_df = pd.DataFrame({
        'Series': short_features.index,
        'Valid_Points_Latest': short_features.values,
        'Valid_Points_PreCOVID': [valid_counts_pre.get(s, 0) for s in short_features.index]
    }).reset_index(drop=True)
    
    display(flagged_df.style.background_gradient(
        subset=['Valid_Points_Latest'], cmap='YlOrRd_r', vmin=0, vmax=MIN_VALID_OBS
    ))
    
    drop_set = set(short_features.index)
    snap_latest = snap_latest[~snap_latest['series_name'].isin(drop_set)]
    snap_pre = snap_pre[~snap_pre['series_name'].isin(drop_set)]
    
    print(f"\n✅ Removed {len(drop_set)} short-history features.")
    print(f"   snap_latest: {snap_latest['series_name'].nunique()} series remaining")
    print(f"   snap_pre:    {snap_pre['series_name'].nunique()} series remaining")
else:
    print(f"✅ All features have >= {MIN_VALID_OBS} valid data points. No filtering needed.")

✅ All features have >= 60 valid data points. No filtering needed.


## 2. Define Data-Type Groupings

Group NOAA series by index type.

Groups:
- **Economic Damage**: NOAA_Economic_Damage_Index and its transformations
- **Human Impact**: NOAA_Human_Impact_Index and its transformations

In [19]:
# Data type prefixes -- order matters (check longer prefixes first)
DATA_TYPE_PREFIXES = [
    ('NOAA_Economic_Damage_Index', 'Economic Damage'),
    ('NOAA_Human_Impact_Index', 'Human Impact'),
]

def classify_series(series_name):
    """Classify a series_name into its data type group."""
    for prefix, group in DATA_TYPE_PREFIXES:
        if series_name.startswith(prefix):
            return group
    return 'Other'

# Get all unique series and classify
all_series = sorted(set(snap_pre['series_name'].unique()) | set(snap_latest['series_name'].unique()))
series_groups = {}
for s in all_series:
    group = classify_series(s)
    series_groups.setdefault(group, []).append(s)

print("Data Type Groups:")
for group, series_list in sorted(series_groups.items()):
    print(f"  {group}: {len(series_list)} series")

Data Type Groups:
  Economic Damage: 96 series
  Human Impact: 96 series


## 3. Helper Functions

In [20]:
def pivot_snapshot(snap_df, series_list):
    """
    Pivot a long-format snapshot to wide format for a specific set of series.
    Takes the LATEST value per date per series (most recent observation).
    """
    mask = snap_df['series_name'].isin(series_list)
    subset = snap_df[mask][['date', 'series_name', 'value']].copy()
    
    subset = subset.drop_duplicates(subset=['date', 'series_name'], keep='last')
    
    wide = subset.pivot(index='date', columns='series_name', values='value')
    wide = wide.sort_index()
    
    wide = wide.dropna(axis=1, how='all')
    wide = wide.dropna(axis=0, how='all')
    
    return wide


def compute_pairwise_correlations(corr_matrix):
    """
    Extract pairwise correlations from a correlation matrix,
    sorted by absolute correlation (descending).
    """
    pairs = []
    cols = corr_matrix.columns
    for i in range(len(cols)):
        for j in range(i+1, len(cols)):
            pairs.append({
                'Feature A': cols[i],
                'Feature B': cols[j],
                'Correlation': corr_matrix.iloc[i, j]
            })
    
    df = pd.DataFrame(pairs)
    df['|Correlation|'] = df['Correlation'].abs()
    df = df.sort_values('|Correlation|', ascending=False).reset_index(drop=True)
    return df


def compute_target_correlations(wide_df, target_series, target_label):
    """
    Compute correlation of each feature with the target MoM change.
    Returns DataFrame sorted by absolute correlation.
    """
    common_dates = wide_df.index.intersection(target_series.index)
    if len(common_dates) < 10:
        return pd.DataFrame(columns=['Feature', f'Corr with {target_label}', '|Correlation|', 'N'])
    
    target_aligned = target_series.loc[common_dates]
    features_aligned = wide_df.loc[common_dates]
    
    results = []
    for col in features_aligned.columns:
        valid = features_aligned[col].notna() & target_aligned.notna()
        if valid.sum() < 10:
            continue
        corr = features_aligned.loc[valid, col].corr(target_aligned[valid])
        results.append({
            'Feature': col,
            f'Corr with {target_label}': corr,
            '|Correlation|': abs(corr),
            'N': int(valid.sum())
        })
    
    df = pd.DataFrame(results)
    if not df.empty:
        df = df.sort_values('|Correlation|', ascending=False).reset_index(drop=True)
    return df

In [21]:
# --- Imports for Advanced Selection ---
import re
import lightgbm as lgb
from sklearn.feature_selection import mutual_info_regression
from sklearn.model_selection import TimeSeriesSplit
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy.cluster import hierarchy
from scipy.stats import spearmanr, pearsonr, binomtest
from tqdm.notebook import tqdm
import gc
import json

# Ensure reproducible randomness
SEED = 42
np.random.seed(SEED)

# --- LightGBM shared params ---
LGB_PARAMS = {
    'objective': 'regression', 'metric': 'l2', 'n_estimators': 100,
    'learning_rate': 0.05, 'num_leaves': 31, 'verbose': -1,
    'n_jobs': 1, 'random_state': SEED,
}

# --- LightGBM column-name sanitization helpers ---
def _sanitize_col(name):
    """Replace characters that LightGBM rejects."""
    return re.sub(r'[^A-Za-z0-9_]', '_', str(name))

def _sanitize_df(df):
    """Return (sanitized_df, clean_to_orig mapping) for LightGBM compat."""
    clean_to_orig = {}
    new_cols = []
    for c in df.columns:
        clean = _sanitize_col(c)
        base = clean
        i = 2
        while clean in clean_to_orig:
            clean = f"{base}_{i}"
            i += 1
        clean_to_orig[clean] = c
        new_cols.append(clean)
    out = df.copy()
    out.columns = new_cols
    return out, clean_to_orig

def _lgb_fit(X, y, params=None):
    if params is None:
        params = LGB_PARAMS
    X_clean, mapping = _sanitize_df(X)
    model = lgb.LGBMRegressor(**params)
    model.fit(X_clean, y)
    return model, mapping

def _lgb_importances(model, mapping):
    clean_names = model.feature_name_
    imp_vals = model.feature_importances_
    return pd.Series({mapping[c]: v for c, v in zip(clean_names, imp_vals)})

def _lgb_predict(model, X, mapping):
    orig_to_clean = {v: k for k, v in mapping.items()}
    X_clean = X.rename(columns=orig_to_clean)
    return model.predict(X_clean)

In [22]:
from scipy.stats import t as t_dist


def _purged_expanding_corr(feature_series, target_series, min_window=60, purge_months=3):
    """
    Compute weighted-average correlation using expanding windows with a purge gap.
    Handles staggered start dates: only uses dates where the feature has data.
    """
    common = feature_series.dropna().index.intersection(target_series.dropna().index)
    common = common.sort_values()
    
    if len(common) < min_window:
        return np.nan, 0
    
    eval_points = np.linspace(min_window, len(common) - 1, min(5, len(common) - min_window), dtype=int)
    eval_points = sorted(set(eval_points))
    
    if not eval_points:
        return np.nan, 0
    
    correlations = []
    window_sizes = []
    for end_idx in eval_points:
        train_end = max(0, end_idx - purge_months)
        if train_end < min_window // 2:
            continue
        
        train_dates = common[:train_end]
        x_train = feature_series.loc[train_dates]
        y_train = target_series.loc[train_dates]
        
        if len(x_train) >= 30 and x_train.std() > 0:
            r = x_train.corr(y_train)
            if not np.isnan(r):
                correlations.append(r)
                window_sizes.append(len(x_train))
    
    if not correlations:
        return np.nan, 0
    
    weights = np.sqrt(window_sizes)
    weighted_corr = np.average(correlations, weights=weights)
    effective_n = int(np.mean(window_sizes))
    
    return weighted_corr, effective_n


def _lgb_screen_group(wide_df, target_series, top_k=15):
    """
    LightGBM-based group screening: catches features useful for tree splits
    even if they have zero marginal correlation with the target.
    """
    common = wide_df.index.intersection(target_series.index)
    if len(common) < 50:
        return []
    
    X = wide_df.loc[common]
    y = target_series.loc[common]
    
    last_date = X.index.max()
    recent_cutoff = last_date - pd.DateOffset(months=3)
    recency_mask = X.apply(lambda col: col.last_valid_index() >= recent_cutoff)
    X = X.loc[:, recency_mask]
    
    if X.shape[1] == 0:
        return []
    
    model, mapping = _lgb_fit(X, y)
    imp = _lgb_importances(model, mapping)
    nonzero = imp[imp > 0].sort_values(ascending=False)
    return nonzero.head(top_k).index.tolist()


def filter_group_data_purged(wide_df, target_series, group_name, vif_threshold=10, fdr_alpha=0.10):
    """
    Group-wise filter with DUAL screening paths:
    
    Path A (Linear): Purged expanding-window correlation + BH-FDR significance
    Path B (Non-linear): LightGBM gain importance pre-screen
    
    The UNION of both paths goes through VIF reduction.
    """
    common = wide_df.index.intersection(target_series.index)
    X_local = wide_df.loc[common].copy()
    y_local = target_series.loc[common]
    
    if X_local.empty:
        return []
    
    last_date = X_local.index.max()
    recent_cutoff = last_date - pd.DateOffset(months=3)
    recency_mask = X_local.apply(lambda col: col.last_valid_index() >= recent_cutoff)
    X_local = X_local.loc[:, recency_mask]
    
    if X_local.shape[1] == 0:
        return []
    
    # === Path A: Purged Expanding-Window Correlation + BH-FDR ===
    feature_pvalues = {}
    for col in X_local.columns:
        avg_corr, eff_n = _purged_expanding_corr(X_local[col], y_local)
        
        if np.isnan(avg_corr) or eff_n < 60:
            continue
        
        t_stat = avg_corr * np.sqrt((eff_n - 2) / (1 - avg_corr**2 + 1e-12))
        p_value = 2 * (1 - t_dist.cdf(abs(t_stat), df=eff_n - 2))
        feature_pvalues[col] = p_value
    
    corr_features = []
    if feature_pvalues:
        sorted_features = sorted(feature_pvalues.items(), key=lambda x: x[1])
        n_tests = len(sorted_features)
        for rank, (feat, pval) in enumerate(sorted_features, 1):
            bh_threshold = (rank / n_tests) * fdr_alpha
            if pval <= bh_threshold:
                corr_features.append(feat)
            else:
                break
    
    # === Path B: LightGBM Gain Importance Pre-Screen ===
    lgb_features = _lgb_screen_group(X_local, y_local, top_k=15)
    
    # === Union of both paths ===
    passed_features = sorted(set(corr_features) | set(lgb_features))
    
    if not passed_features:
        return []
    
    # 3. Iterative Pruning for VIF (need dense matrix)
    X_vif = X_local[passed_features].copy()
    
    target_rows = 100
    current_rows = X_vif.dropna()
    while len(current_rows) < target_rows and X_vif.shape[1] > 1:
        nan_counts = X_vif.isna().sum()
        worst_col = nan_counts.idxmax()
        X_vif = X_vif.drop(columns=[worst_col])
        current_rows = X_vif.dropna()
    
    X_vif = X_vif.dropna()
    y_vif = y_local.loc[X_vif.index]
    
    if len(X_vif) < 50 or X_vif.shape[1] == 0:
        return passed_features[:15]
    
    if X_vif.shape[1] > 60:
        corrs = X_vif.corrwith(y_vif).abs()
        top_60 = corrs.nlargest(60).index
        X_vif = X_vif[top_60]
    
    # 4. VIF Iterative Reduction
    while X_vif.shape[1] > 1:
        try:
            vifs = [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
            max_vif = max(vifs)
            if max_vif <= vif_threshold:
                break
            
            max_idx = vifs.index(max_vif)
            feat_max = X_vif.columns[max_idx]
            
            curr_corr = X_vif.corr().abs()
            partner = curr_corr[feat_max].drop(feat_max).idxmax()
            
            if abs(X_vif[feat_max].corr(y_vif)) < abs(X_vif[partner].corr(y_vif)):
                drop_col = feat_max
            else:
                drop_col = partner
            
            X_vif = X_vif.drop(columns=[drop_col])
        except Exception as e:
            print(f"  [Group {group_name}] VIF Error: {e}")
            break
    
    return X_vif.columns.tolist()

In [23]:
def _block_permute(y, block_size=6, rng=None):
    """Permute target by shuffling contiguous blocks (preserves autocorrelation)."""
    if rng is None:
        rng = np.random.RandomState(SEED)
    
    n = len(y)
    values = y.values if hasattr(y, 'values') else np.array(y)
    
    n_full_blocks = n // block_size
    blocks = [values[i*block_size:(i+1)*block_size] for i in range(n_full_blocks)]
    remainder = values[n_full_blocks*block_size:]
    if len(remainder) > 0:
        blocks.append(remainder)
    
    order = rng.permutation(len(blocks))
    shuffled = np.concatenate([blocks[i] for i in order])
    
    return shuffled[:n]


def get_boruta_importance(X, y, n_runs=100, block_size=6, alpha=0.05):
    """
    Boruta-style feature selection with shadow features.
    Compares each real feature to the 75th percentile of all shadow importances.
    """
    common = X.index.intersection(y.index)
    X_curr = X.loc[common]
    y_curr = y.loc[common]
    
    n_features = X_curr.shape[1]
    feature_names = X_curr.columns.tolist()
    
    hits = np.zeros(n_features, dtype=int)
    
    for run in tqdm(range(n_runs), desc="Boruta Runs", leave=False):
        rng = np.random.RandomState(SEED + run)
        
        shadow_data = {}
        for col in feature_names:
            col_vals = X_curr[col].values.copy()
            valid_mask = ~np.isnan(col_vals) if np.issubdtype(col_vals.dtype, np.floating) else np.ones(len(col_vals), dtype=bool)
            valid_vals = col_vals[valid_mask].copy()
            rng.shuffle(valid_vals)
            col_vals[valid_mask] = valid_vals
            shadow_data[f'_shadow_{col}'] = col_vals
        
        shadow_df = pd.DataFrame(shadow_data, index=X_curr.index)
        X_combined = pd.concat([X_curr, shadow_df], axis=1)
        
        model, mapping = _lgb_fit(X_combined, y_curr)
        importances = _lgb_importances(model, mapping)
        
        shadow_cols = [c for c in importances.index if c.startswith('_shadow_')]
        shadow_threshold = np.percentile(importances[shadow_cols].values, 75)
        
        for i, feat in enumerate(feature_names):
            if importances.get(feat, 0) > shadow_threshold:
                hits[i] += 1
    
    confirmed = []
    tentative = []
    
    for i, feat in enumerate(feature_names):
        p_val = binomtest(hits[i], n=n_runs, p=0.25, alternative='greater').pvalue
        if p_val < alpha:
            confirmed.append(feat)
        elif p_val < alpha * 5:
            tentative.append(feat)
    
    print(f"   Boruta: {len(confirmed)} confirmed, {len(tentative)} tentative "
          f"(hit rates: min={hits.min()}/{n_runs}, max={hits.max()}/{n_runs})")
    
    return confirmed + tentative

In [24]:
def get_vintage_stability(feature_list, target_series, snapshots_dir, min_presence=2):
    """
    Check feature importance stability across historical snapshots.
    Exponential recency weighting: {2010:1, 2014:2, 2018:4, 2022:8, Latest:16}
    """
    years = ['2010', '2014', '2018', '2022', 'Latest']
    weights = {'2010': 1, '2014': 2, '2018': 4, '2022': 8, 'Latest': 16}
    
    scores = pd.DataFrame(np.nan, index=feature_list, columns=years)
    
    for year in years:
        if year == 'Latest':
            snap = snap_latest[snap_latest['series_name'].isin(feature_list)]
        else:
            decade = year[:3] + "0s"
            path = snapshots_dir / decade / year / f"{year}-12.parquet"
            if not path.exists():
                continue
            snap = pd.read_parquet(path)
            snap['date'] = pd.to_datetime(snap['date'])
            snap = snap[snap['series_name'].isin(feature_list)]
        
        if snap.empty:
            continue
        
        X_wide = pivot_snapshot(snap, feature_list)
        X_wide = winsorize_covid_period(X_wide)
        
        common = X_wide.index.intersection(target_series.index)
        if len(common) < 50:
            continue
        
        model, mapping = _lgb_fit(X_wide.loc[common], target_series.loc[common])
        imp = _lgb_importances(model, mapping)
        if imp.sum() > 0:
            imp = imp / imp.sum()
            for feat in imp.index:
                scores.loc[feat, year] = imp[feat]
    
    weight_series = pd.Series(weights)
    
    results = {}
    for feat in feature_list:
        feat_scores = scores.loc[feat].dropna()
        if len(feat_scores) == 0:
            continue
        
        nonzero_count = (feat_scores > 0).sum()
        
        if nonzero_count < min_presence:
            continue
        
        available_weights = weight_series[feat_scores.index]
        weighted_score = (feat_scores * available_weights).sum() / available_weights.sum()
        results[feat] = weighted_score
    
    result_series = pd.Series(results)
    
    latest_scores = scores['Latest'].dropna()
    latest_nonzero = latest_scores[latest_scores > 0].index
    result_series = result_series[result_series.index.isin(latest_nonzero)]
    
    return result_series.sort_values(ascending=False)

In [25]:
def cluster_redundancy(X, feature_list, target_series, max_clusters=50, min_overlap=30):
    """
    Hierarchical clustering to remove redundant features.
    NaN-aware pairwise Spearman correlations.
    """
    if len(feature_list) <= max_clusters:
        return feature_list
    
    X_curr = X[feature_list].copy()
    common = X_curr.index.intersection(target_series.index)
    X_curr = X_curr.loc[common]
    y_curr = target_series.loc[common]
    
    valid_target = y_curr.notna()
    X_curr = X_curr.loc[valid_target]
    y_curr = y_curr.loc[valid_target]
    
    corr = X_curr.corr(method='spearman')
    corr = corr.fillna(0)
    
    for i, col_i in enumerate(feature_list):
        for j, col_j in enumerate(feature_list):
            if i >= j:
                continue
            overlap = (X_curr[col_i].notna() & X_curr[col_j].notna()).sum()
            if overlap < min_overlap:
                corr.loc[col_i, col_j] = 0
                corr.loc[col_j, col_i] = 0
    
    dist_matrix = 1 - corr.abs().values
    np.fill_diagonal(dist_matrix, 0)
    
    dist_matrix = np.maximum(dist_matrix, 0)
    dist_matrix = (dist_matrix + dist_matrix.T) / 2
    
    from scipy.spatial.distance import squareform
    condensed = squareform(dist_matrix, checks=False)
    
    linkage = hierarchy.ward(condensed)
    cluster_ids = hierarchy.fcluster(linkage, t=max_clusters, criterion='maxclust')
    
    X_imputed = X_curr.fillna(X_curr.median())
    mi = mutual_info_regression(X_imputed, y_curr, random_state=SEED)
    mi_series = pd.Series(mi, index=feature_list)
    
    selected = []
    for cluster_id in np.unique(cluster_ids):
        members = [feature_list[i] for i, c in enumerate(cluster_ids) if c == cluster_id]
        best = mi_series[members].idxmax()
        selected.append(best)
    
    return selected

In [26]:
def _extract_split_pairs(model, mapping):
    """Extract parent-child split pairs from LightGBM trees."""
    from collections import Counter
    pair_counts = Counter()
    
    # Map sanitized feature names back to originals
    clean_names = model.feature_name_
    
    booster = model.booster_
    for tree_idx in range(booster.num_trees()):
        tree_info = booster.dump_model()['tree_info'][tree_idx]
        
        def _walk(node, parent_feature=None):
            if 'split_feature' not in node:
                return
            
            feat_idx = node['split_feature']
            if feat_idx < len(clean_names):
                clean_name = clean_names[feat_idx]
                feat_name = mapping.get(clean_name, clean_name)
            else:
                feat_name = f'feature_{feat_idx}'
            
            if parent_feature is not None and parent_feature != feat_name:
                pair = tuple(sorted([parent_feature, feat_name]))
                pair_counts[pair] += 1
            
            if 'left_child' in node:
                _walk(node['left_child'], feat_name)
            if 'right_child' in node:
                _walk(node['right_child'], feat_name)
        
        _walk(tree_info['tree_structure'])
    
    return pair_counts


def interaction_rescue(X, y, confirmed_features, rejected_pool, n_splits=8, gap=3, top_k=10):
    """
    Two-phase interaction rescue:
    Phase 1: Single-feature conditional test
    Phase 2: Split-pair detection from LightGBM trees
    """
    common = X.index.intersection(y.index)
    X_curr = X.loc[common]
    y_curr = y.loc[common]
    
    rejected = [f for f in rejected_pool if f in X_curr.columns and f not in confirmed_features]
    
    if not rejected:
        return []
    
    tscv = TimeSeriesSplit(n_splits=n_splits, gap=gap)
    
    def _cv_mae(feature_set):
        X_sub = X_curr[feature_set]
        maes = []
        for train_idx, test_idx in tscv.split(X_sub):
            model, mapping = _lgb_fit(X_sub.iloc[train_idx], y_curr.iloc[train_idx])
            preds = _lgb_predict(model, X_sub.iloc[test_idx], mapping)
            maes.append(np.mean(np.abs(y_curr.iloc[test_idx].values - preds)))
        return np.mean(maes)
    
    baseline_mae = _cv_mae(confirmed_features)
    
    # === Phase 1: Single-feature conditional rescue ===
    single_improvements = {}
    for feat in tqdm(rejected, desc="Phase 1: Single-feature scan", leave=False):
        trial_mae = _cv_mae(confirmed_features + [feat])
        improvement = (baseline_mae - trial_mae) / (baseline_mae + 1e-12)
        if improvement > 0:
            single_improvements[feat] = improvement
    
    single_rescued = sorted(single_improvements.items(), key=lambda x: x[1], reverse=True)[:top_k]
    
    for feat, imp in single_rescued:
        print(f"   Rescued (single): {feat} (delta MAE = {imp:.4f})")
    
    # === Phase 2: Split-pair detection ===
    all_feats = [f for f in X_curr.columns if f in confirmed_features or f in rejected]
    
    model_full, mapping_full = _lgb_fit(
        X_curr[all_feats], y_curr,
        params={**LGB_PARAMS, 'n_estimators': 200, 'num_leaves': 63, 'max_depth': 6}
    )
    
    pair_counts = _extract_split_pairs(model_full, mapping_full)
    
    rejected_set = set(rejected)
    pair_rescued = []
    
    for (feat_a, feat_b), count in pair_counts.most_common():
        if len(pair_rescued) >= top_k:
            break
        
        new_feats = []
        if feat_a in rejected_set and feat_a not in [f for f, _ in single_rescued]:
            new_feats.append(feat_a)
        if feat_b in rejected_set and feat_b not in [f for f, _ in single_rescued]:
            new_feats.append(feat_b)
        
        if not new_feats:
            continue
        
        trial_feats = confirmed_features + [f for f, _ in single_rescued] + new_feats
        trial_feats = list(dict.fromkeys(trial_feats))
        
        baseline_with_singles = confirmed_features + [f for f, _ in single_rescued]
        baseline_with_singles = list(dict.fromkeys(baseline_with_singles))
        baseline_singles_mae = _cv_mae(baseline_with_singles)
        trial_mae = _cv_mae(trial_feats)
        
        improvement = (baseline_singles_mae - trial_mae) / (baseline_singles_mae + 1e-12)
        if improvement > 0:
            for f in new_feats:
                if f not in [pf for pf, _ in pair_rescued]:
                    pair_rescued.append((f, improvement))
                    print(f"   Rescued (pair w/ {feat_a if f != feat_a else feat_b}, "
                          f"co-occur={count}): {f} (delta MAE = {improvement:.4f})")
    
    all_rescued = [f for f, _ in single_rescued] + [f for f, _ in pair_rescued]
    seen = set()
    result = []
    for f in all_rescued:
        if f not in seen:
            seen.add(f)
            result.append(f)
    
    return result


def sequential_forward_selection(X, y, candidate_features, boruta_hits=None,
                                   n_splits=8, gap=3, min_improvement=0.001,
                                   patience=3, min_features=5):
    """Sequential Forward Selection using walk-forward CV."""
    common = X.index.intersection(y.index)
    X_curr = X.loc[common]
    y_curr = y.loc[common]
    
    tscv = TimeSeriesSplit(n_splits=n_splits, gap=gap)
    
    def _cv_mae(feature_set):
        if not feature_set:
            return float('inf')
        
        X_sub = X_curr[feature_set]
        maes = []
        for train_idx, test_idx in tscv.split(X_sub):
            X_train, X_test = X_sub.iloc[train_idx], X_sub.iloc[test_idx]
            y_train, y_test = y_curr.iloc[train_idx], y_curr.iloc[test_idx]
            
            model, mapping = _lgb_fit(X_train, y_train)
            preds = _lgb_predict(model, X_test, mapping)
            maes.append(np.mean(np.abs(y_test.values - preds)))
        
        return np.mean(maes)
    
    if boruta_hits is not None:
        candidates = sorted(candidate_features, key=lambda f: boruta_hits.get(f, 0), reverse=True)
    else:
        candidates = list(candidate_features)
    
    best_single_mae = float('inf')
    best_single = candidates[0]
    for feat in candidates[:min(20, len(candidates))]:
        mae = _cv_mae([feat])
        if mae < best_single_mae:
            best_single_mae = mae
            best_single = feat
    
    selected = [best_single]
    current_mae = best_single_mae
    remaining = [f for f in candidates if f != best_single]
    consecutive_failures = 0
    
    print(f"   SFS start: {best_single} (MAE={current_mae:.2f})")
    
    while remaining:
        best_next = None
        best_next_mae = current_mae
        
        for feat in remaining:
            trial = selected + [feat]
            mae = _cv_mae(trial)
            if mae < best_next_mae:
                best_next_mae = mae
                best_next = feat
        
        if best_next is None:
            consecutive_failures += 1
            if len(selected) >= min_features and consecutive_failures >= patience:
                print(f"   SFS stop: no improvement for {patience} rounds")
                break
            forced = remaining[0]
            selected.append(forced)
            remaining.remove(forced)
            print(f"   SFS +{forced} (forced, MAE~{current_mae:.2f}, below min_features={min_features})")
            continue
        
        improvement = (current_mae - best_next_mae) / (current_mae + 1e-12)
        
        if improvement < min_improvement:
            consecutive_failures += 1
            if len(selected) >= min_features and consecutive_failures >= patience:
                print(f"   SFS stop: {patience} consecutive improvements < {min_improvement:.4f}")
                break
            selected.append(best_next)
            remaining.remove(best_next)
            current_mae = best_next_mae
            print(f"   SFS +{best_next} (MAE={current_mae:.2f}, delta={improvement:.4f}, "
                  f"patience {consecutive_failures}/{patience})")
        else:
            consecutive_failures = 0
            selected.append(best_next)
            remaining.remove(best_next)
            current_mae = best_next_mae
            print(f"   SFS +{best_next} (MAE={current_mae:.2f}, delta={improvement:.4f})")
    
    print(f"   SFS final: {len(selected)} features, MAE={current_mae:.2f}")
    return selected


# ===========================================================================
#  MAIN PIPELINE — 7 Stages
# ===========================================================================

# --- TARGET VERIFICATION ---
if target_nsa['y_mom'].isna().all() or (target_nsa['y_mom'] == target_nsa['y']).any():
    raise ValueError("Target y_mom does not look like a difference/change variable!")

targets = {'NSA': target_nsa['y_mom'], 'SA': target_sa['y_mom'], 'NSA_Acc': target_nsa['y_acc'], 'SA_Acc': target_sa['y_acc']}
final_results = {}

for t_name, y_target in targets.items():
    print(f"\n{'='*60}")
    print(f"  PIPELINE START: {t_name}")
    print(f"{'='*60}")
    
    # --- COVID Consistency ---
    y_target = winsorize_covid_period(y_target)
    
    # ===== Stage 1: Group-wise Dual Filter (Linear + LightGBM) =====
    stage1_candidates = []
    print("1. Group-wise Dual Filter (Purged Corr + LightGBM screen + VIF)...")
    for group_name, series_list in series_groups.items():
        wide_group = pivot_snapshot(snap_latest, series_list)
        wide_group = winsorize_covid_period(wide_group)
        
        selected = filter_group_data_purged(wide_group, y_target, group_name)
        stage1_candidates.extend(selected)
    
    stage1_candidates = sorted(list(set(stage1_candidates)))
    print(f"   > Stage 1 Survivors: {len(stage1_candidates)} features")
    
    if len(stage1_candidates) == 0:
        print("   WARNING: No features survived Stage 1. Skipping pipeline.")
        final_results[t_name] = []
        continue
    
    # ===== Stage 2: Boruta Importance =====
    print("2. Boruta Feature Selection (shadow features)...")
    X_stage2 = pivot_snapshot(snap_latest, stage1_candidates)
    X_stage2 = winsorize_covid_period(X_stage2)
    
    stage2_candidates = get_boruta_importance(X_stage2, y_target, n_runs=100)
    print(f"   > Stage 2 Survivors: {len(stage2_candidates)} features")
    
    if len(stage2_candidates) == 0:
        print("   WARNING: No features survived Boruta. Falling back to Stage 1 top 30.")
        common = X_stage2.index.intersection(y_target.index)
        corrs = X_stage2.loc[common].corrwith(y_target.loc[common]).abs()
        stage2_candidates = corrs.nlargest(30).index.tolist()
    
    # ===== Stage 3: Vintage Stability =====
    print("3. Vintage Stability (exponential recency weights, min 2 snapshots)...")
    weighted_scores = get_vintage_stability(stage2_candidates, y_target, NOAA_DIR)
    
    if len(weighted_scores) == 0:
        print("   WARNING: No features survived vintage stability. Using all Stage 2.")
        stage3_candidates = stage2_candidates
    else:
        threshold = weighted_scores.median() * 0.5
        stage3_candidates = weighted_scores[weighted_scores > threshold].index.tolist()
    print(f"   > Stage 3 Survivors: {len(stage3_candidates)} features")
    
    # ===== Stage 4: Cluster Redundancy =====
    print("4. Cluster Redundancy Check (NaN-aware Spearman)...")
    stage4_candidates = cluster_redundancy(X_stage2, stage3_candidates, y_target, max_clusters=50)
    print(f"   > Stage 4 Survivors: {len(stage4_candidates)} features")
    
    # ===== Stage 5: Interaction Rescue (two-phase) =====
    print("5. Interaction Rescue (single-feature + split-pair detection)...")
    rescued = interaction_rescue(
        X_stage2, y_target,
        confirmed_features=stage4_candidates,
        rejected_pool=stage1_candidates,
        top_k=10
    )
    stage5_candidates = stage4_candidates + rescued
    seen = set()
    stage5_candidates = [f for f in stage5_candidates if f not in seen and not seen.add(f)]
    print(f"   > Stage 5 Candidates: {len(stage5_candidates)} "
          f"({len(stage4_candidates)} confirmed + {len(rescued)} rescued)")
    
    # ===== Stage 6: Sequential Forward Selection =====
    print("6. Sequential Forward Selection (walk-forward CV with embargo)...")
    final_list = sequential_forward_selection(
        X_stage2, y_target, stage5_candidates,
        n_splits=8, gap=3,
        min_improvement=0.001,
        patience=3,
        min_features=5,
    )
    print(f"   > Final Count: {len(final_list)}")
    
    final_results[t_name] = final_list
    print(f"Selected features ({t_name}):")
    for i, f in enumerate(final_list):
        print(f"  {i+1:2d}. {f}")

# ===== Stage 7: Union MoM + Acc per variant, save 2 files =====
print(f"\n{'='*60}")
print("Merging MoM + Acceleration features (union) and saving...")

merged_results = {}
for variant in ['nsa', 'sa']:
    mom_key = variant.upper()
    acc_key = f"{variant.upper()}_Acc"
    
    mom_feats = set(final_results.get(mom_key, []))
    acc_feats = set(final_results.get(acc_key, []))
    union_feats = sorted(mom_feats | acc_feats)
    
    merged_results[variant] = union_feats
    
    print(f"\n  {variant.upper()}: {len(mom_feats)} MoM + {len(acc_feats)} Acc "
          f"-> {len(union_feats)} union ({len(mom_feats & acc_feats)} overlap)")
    for i, f in enumerate(union_feats):
        src = []
        if f in mom_feats:
            src.append("MoM")
        if f in acc_feats:
            src.append("Acc")
        print(f"    {i+1:2d}. {f}  [{', '.join(src)}]")

for variant, feats in merged_results.items():
    fname = f"selected_features_noaa_{variant}.json"
    with open(PROJECT_ROOT / 'notebooks' / fname, 'w') as f:
        json.dump(feats, f, indent=4)
    print(f"  Saved {fname}: {len(feats)} features")

print("\nDone.")


  PIPELINE START: NSA
1. Group-wise Dual Filter (Purged Corr + LightGBM screen + VIF)...
   > Stage 1 Survivors: 37 features
2. Boruta Feature Selection (shadow features)...


Boruta Runs:   0%|          | 0/100 [00:00<?, ?it/s]

   Boruta: 26 confirmed, 0 tentative (hit rates: min=0/100, max=100/100)
   > Stage 2 Survivors: 26 features
3. Vintage Stability (exponential recency weights, min 2 snapshots)...
   > Stage 3 Survivors: 26 features
4. Cluster Redundancy Check (NaN-aware Spearman)...
   > Stage 4 Survivors: 26 features
5. Interaction Rescue (single-feature + split-pair detection)...


Phase 1: Single-feature scan:   0%|          | 0/11 [00:00<?, ?it/s]

   Rescued (single): NOAA_Human_Impact_Index_pct_chg_zscore_12m_lag_12m (delta MAE = 0.0060)
   Rescued (single): NOAA_Economic_Damage_Index_chg_12m_lag_3m (delta MAE = 0.0022)
   Rescued (single): NOAA_Economic_Damage_Index_zscore_12m (delta MAE = 0.0014)
   Rescued (pair w/ NOAA_Economic_Damage_Index_chg_6m_lag_18m, co-occur=10): NOAA_Economic_Damage_Index_zscore_12m_lag_6m (delta MAE = 0.0007)
   Rescued (pair w/ NOAA_Human_Impact_Index_zscore_3m_lag_6m, co-occur=4): NOAA_Economic_Damage_Index_chg_12m_lag_12m (delta MAE = 0.0039)
   Rescued (pair w/ NOAA_Economic_Damage_Index_chg_12m_lag_12m, co-occur=4): NOAA_Human_Impact_Index_zscore_3m_lag_6m (delta MAE = 0.0039)
   > Stage 5 Candidates: 32 (26 confirmed + 6 rescued)
6. Sequential Forward Selection (walk-forward CV with embargo)...
   SFS start: NOAA_Human_Impact_Index_chg_3m_lag_6m (MAE=747.48)
   SFS +NOAA_Human_Impact_Index_zscore_3m_lag_18m (MAE=736.16, delta=0.0151)
   SFS +NOAA_Human_Impact_Index_pct_chg_rolling_std_6m_lag_

...

   Rescued (single): NOAA_Economic_Damage_Index_zscore_12m_lag_12m (delta MAE = 0.0065)
   Rescued (single): NOAA_Economic_Damage_Index_chg_12m_lag_3m (delta MAE = 0.0056)
   Rescued (single): NOAA_Economic_Damage_Index_chg_12m_lag_1m (delta MAE = 0.0028)
   Rescued (single): NOAA_Economic_Damage_Index_zscore_3m_lag_6m (delta MAE = 0.0015)
   > Stage 5 Candidates: 27 (23 confirmed + 4 rescued)
6. Sequential Forward Selection (walk-forward CV with embargo)...
   SFS start: NOAA_Human_Impact_Index_chg_3m_lag_1m (MAE=1040.34)
   SFS +NOAA_Human_Impact_Index_rolling_mean_3m_lag_18m (forced, MAE~1040.34, below min_features=5)
   SFS +NOAA_Economic_Damage_Index_chg_6m_lag_3m (forced, MAE~1040.34, below min_features=5)
   SFS +NOAA_Human_Impact_Index_zscore_3m_lag_12m (forced, MAE~1040.34, below min_features=5)
   SFS +NOAA_Economic_Damage_Index_chg_3m_lag_3m (forced, MAE~1040.34, below min_features=5)
   SFS stop: 3 consecutive improvements < 0.0010
   SFS final: 5 features, MAE=1040.34
   >

Boruta Runs:   0%|          | 0/100 [00:00<?, ?it/s]

   Boruta: 19 confirmed, 0 tentative (hit rates: min=1/100, max=100/100)
   > Stage 2 Survivors: 19 features
3. Vintage Stability (exponential recency weights, min 2 snapshots)...
   > Stage 3 Survivors: 19 features
4. Cluster Redundancy Check (NaN-aware Spearman)...
   > Stage 4 Survivors: 19 features
5. Interaction Rescue (single-feature + split-pair detection)...


Phase 1: Single-feature scan:   0%|          | 0/9 [00:00<?, ?it/s]

   Rescued (single): NOAA_Human_Impact_Index_chg_12m_lag_18m (delta MAE = 0.0139)
   Rescued (single): NOAA_Economic_Damage_Index_chg_12m_lag_3m (delta MAE = 0.0124)
   Rescued (single): NOAA_Human_Impact_Index_chg_12m_lag_12m (delta MAE = 0.0046)
   Rescued (single): NOAA_Economic_Damage_Index_rolling_std_6m_lag_18m (delta MAE = 0.0031)
   Rescued (single): NOAA_Economic_Damage_Index_zscore_12m (delta MAE = 0.0029)
   Rescued (pair w/ NOAA_Human_Impact_Index_pct_chg_rolling_std_6m_lag_6m, co-occur=33): NOAA_Economic_Damage_Index_diff_zscore_3m_lag_18m (delta MAE = 0.0003)
   Rescued (pair w/ NOAA_Economic_Damage_Index_zscore_12m, co-occur=14): NOAA_Human_Impact_Index_zscore_3m_lag_3m (delta MAE = 0.0177)
   Rescued (pair w/ NOAA_Economic_Damage_Index_zscore_12m, co-occur=9): NOAA_Economic_Damage_Index_pct_chg_rolling_std_6m_lag_18m (delta MAE = 0.0026)
   > Stage 5 Candidates: 27 (19 confirmed + 8 rescued)
6. Sequential Forward Selection (walk-forward CV with embargo)...
   SFS start: